1. Введіть своє прізвище та ім'я, а також номер варіанту

In [ ]:
# =========================
# 0. ДАНІ СТУДЕНТА
# =========================
student_name = "Прізвище ім'я"
variant_number = 0


2. Запустіть код, натисніть кнопку "Вибрати файли" та виберіть тренувальний та тестовий датасети, отримані в результаті ЛР2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc,
    f1_score
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import files
import pandas as pd

# =========================
# 1. ЗАВАНТАЖЕННЯ ДАНИХ
# =========================

# Завантаження файлів
uploaded = files.upload()  # відкриється діалог для вибору файлів

# Зчитування CSV у DataFrame
train_df = pd.read_csv("train_processed.csv")
test_df = pd.read_csv("test_processed.csv")

# Перевірка
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values

X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values


3. Аналіз датасетів, навчання нейронної мережі, передбачення класів, розрахунок метрик, їхня графічна візуалізація та генерація .png та .csv файлів

In [ ]:
# =========================
# 2. КЛАСИ
# =========================
class_names = ["Normal", "DoS", "Probe", "R2L", "U2R"]
num_classes = len(class_names)


# =========================
# ⚖️ БАЛАНС КЛАСІВ
# =========================
class_counts = np.bincount(y_train.astype(int))
total = len(y_train)

class_weight = {
    i: total / (num_classes * class_counts[i])
    for i in range(num_classes)
}

# =========================
# 3. МОДЕЛЬ
# =========================
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# =========================
# РАННЯ ЗУПИНКА
# =========================
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


# =========================
# 4. НАВЧАННЯ
# =========================
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)


# =========================
# 5. ПЕРЕДБАЧЕННЯ
# =========================
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# =========================
# ЗБЕРЕЖЕННЯ ПОКЛАСОВИХ ПЕРЕДБАЧЕНЬ
# =========================

# створюємо копію тестового датасету
results_df = test_df.copy()

# додаємо справжні та передбачені класи
results_df["true_class"] = y_test
results_df["predicted_class"] = y_pred

# позначаємо правильність передбачення
results_df["correct"] = (results_df["true_class"] == results_df["predicted_class"]).astype(int)

# якщо класи числові — можна додати текстові мітки (опціонально)
results_df["true_label"] = results_df["true_class"].apply(lambda x: class_names[int(x)])
results_df["pred_label"] = results_df["predicted_class"].apply(lambda x: class_names[int(x)])

# збереження
pred_file = f"predictions_{student_name.replace(' ', '_')}_Варіант_{variant_number}.csv"
f"ЛР3 - {student_name} - Варіант {variant_number}.png"
results_df.to_csv(pred_file, index=False)

print(f"\nЗбережено файл передбачень: {pred_file}")

# =========================
# АНАЛІЗ ПОМИЛОК ПЕРЕДБАЧЕННЯ
# =========================

# фільтруємо неправильні передбачення
misclassified_df = results_df[results_df["correct"] == 0].copy()

# збереження окремого файлу тільки з помилками
mis_file = f"misclassified_{student_name.replace(' ', '_')}_var{variant_number}.csv"
misclassified_df.to_csv(mis_file, index=False)

print(f"\nЗбережено файл помилок: {mis_file}")
print(f"Кількість помилкових передбачень: {len(misclassified_df)}")


# =========================
# ВИВІД ПРИКЛАДІВ ПОМИЛОК
# =========================

print("\n🔎 Приклади помилкових передбачень (до 10 рядків):\n")

cols_to_show = ["true_label", "pred_label"]

# якщо є багато ознак — не виводимо їх повністю, щоб не засмічувати консоль
print(misclassified_df[cols_to_show].head(10))
print("\nІндекси помилкових рядків у тестовому наборі:")
print(misclassified_df.index[:20].tolist())

# =========================
# 📊 ROC-AUC
# =========================
y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))

roc_auc = roc_auc_score(
    y_test_bin,
    y_pred_prob,
    multi_class="ovr",
    average=None  # <-- ключове
)


# =========================
# 📊 F1-МІРА
# =========================
f1_macro = f1_score(y_test, y_pred, average='macro')


# =========================
# 📄 ЗВІТ
# =========================
print("\n Classification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=class_names
))

print("\n ROC-AUC по класах:")

for i, score in enumerate(roc_auc):
    print(f"{class_names[i]}: {score:.4f}")

print(f"F1 Macro: {f1_macro:.4f}")


# =========================
# RESULTS.CSV
# =========================
metrics = pd.DataFrame([{
    "student": student_name,
    "variant": variant_number,
    "accuracy": np.mean(y_pred == y_test),
    "roc_auc": roc_auc,
    "f1_macro": f1_macro,
    "train_loss": history.history["loss"][-1],
    "val_loss": history.history["val_loss"][-1],
    "val_accuracy": history.history["val_accuracy"][-1]
}])

metrics.to_csv(
    f"results_{student_name.replace(' ', '_')}_Варіант_{variant_number}.csv",
    index=False
)

# =========================
# 6. ROC-криві
# =========================
def plot_roc(ax):
    y_test_bin_local = label_binarize(y_test, classes=np.arange(num_classes))

    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_test_bin_local[:, i], y_pred_prob[:, i])
        roc_auc_i = auc(fpr, tpr)

        ax.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc_i:.6f})")

    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_title("ROC-криві")
    ax.set_xlabel("Хибнопозитивні спрацювання")
    ax.set_ylabel("Справжньопозитивні спрацювання")
    ax.legend()

print("\nПеревірка AUC по класах (без округлення):\n")

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    auc_i = auc(fpr, tpr)

    print(f"{class_names[i]}: {auc_i:.10f}")


# =========================
# 7. ВІЗУАЛІЗАЦІЯ 2x2
# =========================
def plot_results(history, y_true, y_pred):

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    fig.suptitle(
        f"ЛР3 — Навчання нейромережі\n"
        f"Студент: {student_name} / Варіант: {variant_number}",
        fontsize=16
    )

    ax1, ax2, ax3, ax4 = axes.ravel()

    # =====================
    # ВТРАТИ
    # =====================
    ax1.plot(history.history['loss'], label='Тренувальна помилка')
    ax1.plot(history.history['val_loss'], label='Валідаційна помилка')
    ax1.set_title("Втрати моделі")
    ax1.set_xlabel("Епоха")
    ax1.set_ylabel("Втрати")
    ax1.legend()

    # =====================
    # ТОЧНІСТЬ
    # =====================
    ax2.plot(history.history['accuracy'], label='Тренувальна точність')
    ax2.plot(history.history['val_accuracy'], label='Валідаційна точність')
    ax2.set_title("Точність моделі")
    ax2.set_xlabel("Епоха")
    ax2.set_ylabel("Точність")
    ax2.legend()

    # =====================
    # НОРМАЛІЗОВАНА МАТРИЦЯ ПОМИЛОК
    # =====================
    cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
    ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
        ax=ax3, cmap="Blues", colorbar=False
    )
    ax3.set_title("Нормалізована матриця помилок")
    ax3.set_xlabel("Передбачені класи")
    ax3.set_ylabel("Справжні класи")

    # =====================
    # ROC-КРИВІ
    # =====================
    plot_roc(ax4)

    plt.tight_layout()

    save_path = f"ЛР3_{student_name.replace(' ', '_')}_Варіант_{variant_number}.png"
    plt.savefig(save_path, dpi=300)

    plt.show()


plot_results(history, y_test, y_pred)
